In [ ]:
import pandas as pd
import numpy as np

# ────────────────────────────────────────────────
#   SETTINGS – change these as needed
# ────────────────────────────────────────────────

FILE_PATH = "Historical Data (1).xlsx"          # ← your file name here
SHEET_NAME = "Sheet1"

# What counts as a "repeater" (minimum appearances in data)
REPEATER_MIN_RUNS = 3

# Threshold: if planned covers < 70% of buffer days → likely low stock
LOW_STOCK_THRESHOLD = 0.7

# ────────────────────────────────────────────────

def load_and_prepare_data(path, sheet):
    df = pd.read_excel(path, sheet_name=sheet)

    # Rename messy column if needed
    if "Actual Cycle Std Cycle Time per running Cavity" in df.columns:
        df = df.rename(columns={
            "Actual Cycle Std Cycle Time per running Cavity": "Actual Cycle"
        })

    # Select useful columns
    keep = [
        'Part', 'Monthly Requirement', 'Daily Requirement', 'Buffer Requirement',
        'No of Knaban Cards', 'Planned Quantity', 'Qty/ bin',
        'Actual Production', 'Setup Rejections', 'Actual Rejections'
    ]
    keep = [c for c in keep if c in df.columns]
    df = df[keep].copy()

    # Clean numeric columns
    for col in ['Monthly Requirement', 'Daily Requirement', 'Buffer Requirement',
                'Planned Quantity', 'Qty/ bin', 'No of Knaban Cards',
                'Actual Production']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    df = df.dropna(subset=['Part', 'Planned Quantity', 'Daily Requirement'])

    return df


def analyze_parts(df):
    summary = []

    grouped = df.groupby('Part')

    for part, g in grouped:
        monthly = g['Monthly Requirement'].iloc[0] if 'Monthly Requirement' in g else np.nan
        daily = g['Daily Requirement'].iloc[0] if 'Daily Requirement' in g else np.nan
        buffer = g['Buffer Requirement'].iloc[0] if 'Buffer Requirement' in g else np.nan

        runs = len(g)
        total_planned = g['Planned Quantity'].sum()
        avg_planned = g['Planned Quantity'].mean()

        if daily > 0 and not pd.isna(daily):
            avg_days_covered = avg_planned / daily
        else:
            avg_days_covered = np.nan

        # Rough low-stock flag
        if buffer > 0 and daily > 0:
            buffer_days = buffer / daily
            avg_planned_vs_buffer = avg_planned / buffer if buffer > 0 else np.nan
            low_stock_likely = avg_planned_vs_buffer < LOW_STOCK_THRESHOLD
        else:
            buffer_days = np.nan
            low_stock_likely = False

        category = "Repeater" if runs >= REPEATER_MIN_RUNS else "Stranger / Infrequent"

        summary.append({
            'Part': part,
            'Monthly Req': round(monthly) if not pd.isna(monthly) else '-',
            'Daily Req': round(daily, 1) if not pd.isna(daily) else '-',
            'Buffer Days': round(buffer_days, 1) if not pd.isna(buffer_days) else '-',
            'Runs in data': runs,
            'Category': category,
            'Avg planned qty': round(avg_planned) if not pd.isna(avg_planned) else '-',
            'Avg days covered': round(avg_days_covered, 1) if not pd.isna(avg_days_covered) else '-',
            'Likely low stock when planned?': 'Yes' if low_stock_likely else 'No / unclear',
            'Total planned this period': round(total_planned)
        })

    result = pd.DataFrame(summary)
    result = result.sort_values(['Runs in data', 'Monthly Req'], ascending=[False, False])

    return result


# ────────────────────────────────────────────────
#   Run
# ────────────────────────────────────────────────

df = load_and_prepare_data(FILE_PATH, SHEET_NAME)
summary_df = analyze_parts(df)

print("\nPart Planning Behavior Summary\n")
print(summary_df.to_string(index=False))
print("\n")

# Optional: save to CSV
# summary_df.to_csv("part_planning_summary.csv", index=False)